### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots aren’t “talking” in the human sense—they’re **vocal mimics** that have evolved a remarkable ability to copy sounds they hear. Several biological and social factors combine to make this possible:\n\n| Factor | How it helps parrots mimic speech |\n|--------|-----------------------------------|\n| **Vocal learning ability** | Unlike most birds, parrots belong to a rare group of “vocal learners” (the others are songbirds, hummingbirds, and some mammals). Their brain contains a specialized vocal‑motor pathway (the **song system** and its counterpart, the **parrot vocal learning circuit**) that lets them hear a sound, store a neural template, and reproduce it with precise control of the syrinx (the bird’s vocal organ). |\n| **Social nature** | In the wild, parrots live in tight‑knit flocks where they constantly exchange calls to keep the group together, signal food sources, or warn of danger. Mimicking sounds is a way to bond, establish hierarchy, and maintain cohe

In [2]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools = model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to get weather for Boston using function. Use function call.', 'tool_calls': [{'id': 'fc_1735e3b9-179d-41b2-8bb4-7b2109836463', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 127, 'total_tokens': 169, 'completion_time': 0.088656214, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.026235038, 'prompt_tokens_details': None, 'queue_time': 0.352861451, 'total_time': 0.114891252}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8ea50c0161', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a13080-705a-7882-a076-4a4f48099a86-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_1735e3b9-179d-41b2-8bb4-7b2109836463', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'i

### Tool Execution Loops

In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny right now. Enjoy the clear skies!


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "What\'s the weather in Boston?" We need to get weather. Use function get_weather with location "Boston".', 'tool_calls': [{'id': 'fc_9afc1afd-ae5e-420a-b964-a66a6e1f4c5b', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 126, 'total_tokens': 179, 'completion_time': 0.111418965, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.005650634, 'prompt_tokens_details': None, 'queue_time': 0.383639165, 'total_time': 0.117069599}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_90620edd96', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a13082-7f89-7042-aae0-a9edb6429132-0', tool_calls=[{'name': 'get_weather', 'args': {'locatio